In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define start and end dates
start_date = "2020-01-01"
end_date = "2030-12-31"

# Generate continuous dates
dim_date_df = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{start_date}'),
        to_date('{end_date}'),
        interval 1 day
    )
) AS full_date
""")

# Add derived columns
dim_date_df = dim_date_df \
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int")) \
    .withColumn("day", dayofmonth(col("full_date"))) \
    .withColumn("month", month(col("full_date"))) \
    .withColumn("month_name", date_format(col("full_date"), "MMMM")) \
    .withColumn("short_month", date_format(col("full_date"), "MMM")) \
    .withColumn("quarter", quarter(col("full_date"))) \
    .withColumn("year", year(col("full_date"))) \
    .withColumn("week_of_year", weekofyear(col("full_date"))) \
    .withColumn("day_of_week", date_format(col("full_date"), "E")) \
    .withColumn("weekday_name", date_format(col("full_date"), "EEEE")) \
    .withColumn(
        "weekend_flag",
        when(dayofweek(col("full_date")).isin(1,7), "Y")
        .otherwise("N")
    ) \
    .withColumn(
        "fiscal_year",
        when(month(col("full_date")) >= 4,
             concat(lit("FY-"), year(col("full_date")) + 1))
        .otherwise(concat(lit("FY-"), year(col("full_date"))))
    )

# Display dataframe
display(dim_date_df)

In [0]:
dim_date_df.write.format("delta").mode("overwrite").option("header","true").save("abfss://gold@retailstorage1881.dfs.core.windows.net/dim_date/")